# Stage 2 — Feature Engineering

Builds the (household, campaign) modelling table for the three Type A campaigns
(8, 13, 18): one row per household that had at least one purchase in the 16-week
window before that campaign's start date, with features computed only from that
pre-period, plus the propensity label (`treated`, `redeemed`) and the uplift outcomes
(`outcome_spend`, `outcome_coupon_spend`) measured during the campaign window.

Reusable feature/label logic lives in [`src/features.py`](../src/features.py); this
notebook just loads the raw tables, calls it, saves the result, and checks it.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.features import build_model_table

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load raw tables

In [2]:
raw = {path.stem: pd.read_parquet(path) for path in RAW_DIR.glob("*.parquet")}
{name: df.shape for name, df in raw.items()}


{'campaigns': (6589, 2),
 'campaign_descriptions': (27, 4),
 'coupons': (116204, 3),
 'coupon_redemptions': (2102, 4),
 'demographics': (801, 8),
 'products': (92331, 7),
 'promotions': (20940529, 5),
 'transactions': (1469307, 11)}

## 2. Build the modelling table

`build_model_table` does the following for each of campaigns 8, 13, and 18:

1. Takes the 16-week window before the campaign's `start_date` and filters transactions
   to strictly inside it (`window_start <= transaction_timestamp < start_date`).
2. Defines the population as every household with at least one transaction in that window.
3. Computes all features from that pre-period only, and asserts the max transaction
   timestamp used is still before `start_date`.
4. Joins demographics (one-hot, with a `has_demographics` flag) and computes labels from
   the campaign window itself (`start_date` through the end of `end_date`'s calendar day).

`spend_trend_ratio`'s 1st/99th percentile cap is a special case: it's fit **only on
campaigns 8 and 13** (the training campaigns) and then applied to all three campaigns,
including campaign 18. Campaign 18 is the out-of-time test set, so no statistic used to
build its features — not even an outlier cap — is allowed to be derived from its own
data.

In [3]:
model_table, top_departments, trend_cap_bounds = build_model_table(raw, campaign_ids=(8, 13, 18))

print("Top 6 departments by total sales value:", top_departments)
print("spend_trend_ratio cap bounds (fit on campaigns 8, 13):", trend_cap_bounds)
print("Model table shape:", model_table.shape)
model_table.head()


Top 6 departments by total sales value: ['GROCERY', 'DRUG GM', 'FUEL', 'PRODUCE', 'MEAT', 'MEAT-PCKGD']
spend_trend_ratio cap bounds (fit on campaigns 8, 13): (np.float64(0.0), np.float64(14.681122026126351))
Model table shape: (7015, 70)


,campaign_id,household_id,recency_days,frequency_baskets,monetary_spend,avg_basket_value,items_per_basket,retail_disc_share,coupon_disc_total,campaign_coupon_spend_share,spend_share_grocery,spend_share_drug_gm,spend_share_fuel,spend_share_produce,spend_share_meat,spend_share_meat_pckgd,past_redemptions_count,earlier_campaigns_count,concurrent_campaigns_count,age_19-24,age_25-34,age_35-44,age_45-54,age_55-64,age_65+,age_nan,income_100-124K,income_125-149K,income_15-24K,income_150-174K,income_175-199K,income_200-249K,income_25-34K,income_250K+,income_35-49K,income_50-74K,income_75-99K,income_Under 15K,income_nan,home_ownership_Homeowner,home_ownership_Probable Homeowner,home_ownership_Probable Renter,home_ownership_Renter,home_ownership_nan,marital_status_Married,marital_status_Unmarried,marital_status_nan,household_size_1,household_size_2,household_size_3,household_size_4,household_size_5+,household_size_nan,household_comp_1 Adult Kids,household_comp_1 Adult No Kids,household_comp_2 Adults Kids,household_comp_2 Adults No Kids,household_comp_nan,kids_count_0,kids_count_1,kids_count_2,kids_count_3+,kids_count_nan,has_demographics,spend_trend_ratio,trend_missing,treated,redeemed,outcome_spend,outcome_coupon_spend
0,8,1,11,13,670.65,51.588462,25.153846,0.155580,19.78,0.336852,0.628674,0.080981,0.0,0.118139,0.000000,0.105346,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0.502333,0,1,1,429.01,95.51
1,8,2,65,3,167.45,55.816667,20.000000,0.147626,1.00,0.187996,0.593073,0.254285,0.0,0.008898,0.000000,0.017856,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0,0,0,72.78,35.86
2,8,3,3,7,576.41,82.344286,51.571429,0.243802,30.24,0.516282,0.726202,0.032807,0.0,0.024548,0.103069,0.107406,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.553626,0,1,0,121.54,55.76
3,8,4,33,9,136.09,15.121111,5.111111,0.122272,0.00,0.384966,0.620692,0.121317,0.0,0.000000,0.054229,0.150268,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0,1,0,30.02,15.94
4,8,5,76,4,56.17,14.042500,7.000000,0.109489,0.00,0.311020,0.392380,0.154531,0.0,0.038989,0.049671,0.160050,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0,0,0,87.56,53.23


## 3. Save to `data/processed/model_table.parquet`

In [4]:
model_table.to_parquet(PROCESSED_DIR / "model_table.parquet", index=False)
sorted(p.name for p in PROCESSED_DIR.glob("*.parquet"))


['model_table.parquet']

## 4. Row counts and treated rate per campaign

`treated` is 1 if the household appears in the `campaigns` table for that campaign.
Counts are a little below the raw targeted counts from Stage 1 (1,076 / 1,077 / 1,133)
because a handful of targeted households had no purchase in the 16-week pre-period and
are excluded from the population.


In [5]:
campaign_counts = model_table.groupby("campaign_id").agg(
    n_households=("household_id", "size"),
    treated_rate=("treated", "mean"),
)
campaign_counts


,n_households,treated_rate
campaign_id,,
8,2320,0.463362
13,2351,0.457252
18,2344,0.481655


## 5. Redemption rate among treated households

In [6]:
treated_only = model_table[model_table["treated"] == 1]
redemption_rate = treated_only.groupby("campaign_id")["redeemed"].mean().rename("redemption_rate")
redemption_rate


campaign_id
8     0.146977
13    0.182326
18    0.189548
Name: redemption_rate, dtype: float64

## 5b. Model table counts vs. raw Stage 1 counts

Checks how many targeted households and redeemers survive the "at least one purchase in
the 16-week pre-period" population filter, and confirms the redemption rate above is
computed on that filtered population (redeemers / treated households *in the model
table*), not on the raw Stage 1 targeted/redeeming counts.

In [7]:
raw_targeted = raw["campaigns"].groupby("campaign_id")["household_id"].nunique()
raw_redeemers = raw["coupon_redemptions"].groupby("campaign_id")["household_id"].nunique()

model_treated = model_table.groupby("campaign_id")["treated"].sum()
model_redeemers = treated_only.groupby("campaign_id")["redeemed"].sum()

coverage = pd.DataFrame(
    {
        "raw_targeted_households": raw_targeted,
        "model_table_treated_households": model_treated,
        "targeted_dropped_by_population_filter": raw_targeted - model_treated,
        "raw_redeeming_households": raw_redeemers,
        "model_table_redeemers": model_redeemers,
        "redeemers_dropped_by_population_filter": raw_redeemers - model_redeemers,
    }
).loc[[8, 13, 18]]
coverage["redemption_rate_on_filtered_population"] = (
    coverage["model_table_redeemers"] / coverage["model_table_treated_households"]
)
coverage["redemption_rate_on_raw_population"] = (
    coverage["raw_redeeming_households"] / coverage["raw_targeted_households"]
)
coverage


,raw_targeted_households,model_table_treated_households,targeted_dropped_by_population_filter,raw_redeeming_households,model_table_redeemers,redeemers_dropped_by_population_filter,redemption_rate_on_filtered_population,redemption_rate_on_raw_population
campaign_id,,,,,,,,
8,1076,1075.0,1.0,158.0,158.0,0.0,0.146977,0.146840
13,1077,1075.0,2.0,196.0,196.0,0.0,0.182326,0.181987
18,1133,1129.0,4.0,214.0,214.0,0.0,0.189548,0.188879


The population filter drops a small number of targeted households (1 / 2 / 4 for
campaigns 8 / 13 / 18) that had no transaction in the 16-week pre-period, so no features
could be built for them. It drops **zero** redeemers in all three campaigns — every
household that redeemed a coupon also had pre-period purchase activity. Since only
non-redeeming treated households are ever dropped, the filtered-population redemption
rate is always slightly *higher* than the raw rate (e.g. campaign 18: 18.95% filtered vs.
18.89% raw) — confirming the rate reported in section 5 is `redeemers / treated` computed
within the model table, not against the original Stage 1 targeted count.

## 6. `describe()` of features

Excludes identifiers and labels so the describe table is all modelling features.


In [8]:
label_cols = ["campaign_id", "household_id", "treated", "redeemed", "outcome_spend", "outcome_coupon_spend"]
feature_cols = [c for c in model_table.columns if c not in label_cols]

model_table[feature_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
recency_days,7015.0,12.509765,19.842612,0.00,1.000000,4.000000,14.000000,111.000000
frequency_baskets,7015.0,20.560228,23.554438,1.00,6.000000,14.000000,27.000000,309.000000
monetary_spend,7015.0,598.290644,655.689680,0.59,140.235000,389.150000,822.405000,8166.260000
avg_basket_value,7015.0,32.000679,23.615320,0.59,16.099000,26.056176,41.349091,234.180000
items_per_basket,7015.0,730.092812,1526.030930,1.00,8.844130,19.250000,821.277778,19112.000000
...,...,...,...,...,...,...,...,...
kids_count_3+,7015.0,0.029223,0.168443,0.00,0.000000,0.000000,0.000000,1.000000
kids_count_nan,7015.0,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000
has_demographics,7015.0,0.341554,0.474265,0.00,0.000000,0.000000,1.000000,1.000000
spend_trend_ratio,7015.0,1.329009,1.985692,0.00,0.397512,0.897195,1.419579,14.681122


## 7. Treated vs. control feature comparison — campaign 18

Mean of each feature for treated vs. control households, plus the standardized mean
difference (SMD = difference in means / pooled standard deviation across both groups).
SMD lets us compare features on wildly different scales (a 0-1 share vs. a dollar amount)
on one common footing — anything above ~0.2 is usually considered a meaningfully
imbalanced covariate in causal inference.


In [9]:
campaign_18 = model_table[model_table["campaign_id"] == 18]

comparison = campaign_18.groupby("treated")[feature_cols].mean().T
comparison.columns = ["control_mean", "treated_mean"]
comparison["diff"] = comparison["treated_mean"] - comparison["control_mean"]

pooled_std = campaign_18[feature_cols].std()
comparison["smd"] = (comparison["diff"] / pooled_std.replace(0, np.nan)).round(3)

comparison = comparison.round(3).sort_values("smd", key=lambda s: s.abs(), ascending=False)
comparison.head(20)


,control_mean,treated_mean,diff,smd
earlier_campaigns_count,0.703,3.257,2.554,1.192
has_demographics,0.130,0.570,0.439,0.926
monetary_spend,305.152,897.576,592.424,0.914
frequency_baskets,11.072,30.541,19.469,0.839
concurrent_campaigns_count,0.055,0.500,0.444,0.755
kids_count_0,0.073,0.376,0.302,0.731
home_ownership_Homeowner,0.084,0.356,0.272,0.662
recency_days,19.064,6.041,-13.023,-0.656
household_size_2,0.044,0.235,0.191,0.558
marital_status_Married,0.051,0.246,0.195,0.554


## 8. Feature glossary

| Feature | What it measures |
|---|---|
| `recency_days` | Days between the household's last pre-period purchase and the campaign start date — lower means they shopped more recently |
| `frequency_baskets` | Number of distinct shopping trips (baskets) in the 16-week pre-period |
| `monetary_spend` | Total spend in the pre-period |
| `avg_basket_value` | `monetary_spend / frequency_baskets` — average trip size in dollars |
| `items_per_basket` | Average number of items bought per trip |
| `spend_trend_ratio` | (avg weekly spend in the most recent 4 weeks) / (avg weekly spend in the prior 12 weeks) — above 1 means spend is accelerating into the campaign. Undefined cases (zero spend in the prior-12-week sub-window) are filled with 1.0 ("no change"); the ratio is capped at the 1st/99th percentile of the **training campaigns' (8, 13) own distribution**, and that same cap is applied to campaign 18 so no statistic is fit on the held-out test campaign |
| `trend_missing` | 1 if `spend_trend_ratio` was undefined (zero prior-12-week spend) and filled with 1.0, 0 otherwise |
| `retail_disc_share` | Share of pre-period spend that came from a non-coupon retail discount (shelf markdowns, etc.) |
| `coupon_disc_total` | Total coupon discount dollars redeemed in the pre-period, across any coupon |
| `campaign_coupon_spend_share` | Share of pre-period spend already on products that *this* campaign's coupons cover — a proxy for whether the household already buys the couponed category |
| `spend_share_<department>` | Share of pre-period spend in each of the 6 largest departments by total sales (grocery, drug/GM, fuel, produce, meat, packaged meat) |
| `past_redemptions_count` | Count of coupon redemptions, any campaign, strictly before this campaign's start date |
| `earlier_campaigns_count` | Number of distinct campaigns (other than this one) that targeted this household and started before this campaign |
| `concurrent_campaigns_count` | Number of other campaigns targeting this household that were still active (start ≤ this start ≤ end) when this campaign began |
| `age_*`, `income_*`, `home_ownership_*`, `marital_status_*`, `household_size_*`, `household_comp_*`, `kids_count_*` | One-hot demographic indicators; a `_nan` column appears when the household has a demographics record but left that specific field blank |
| `has_demographics` | 1 if the household has any demographics record at all (only ~32% do, per Stage 1) |

**Labels:**

| Label | What it measures |
|---|---|
| `treated` | 1 if the household was targeted by this campaign |
| `redeemed` | 1 if the household redeemed a coupon for this campaign (structurally 0 for all control households) |
| `outcome_spend` | Total spend during the campaign window (start date through end of end date) |
| `outcome_coupon_spend` | Spend on this campaign's couponed products during the campaign window — the uplift outcome, since it's measurable for both treated and control households |

## 9. What the treated vs. control comparison tells us

Targeting for campaign 18 was clearly **not random**. The largest gaps (by SMD) are:

- **Prior campaign/coupon history** (`earlier_campaigns_count`, `past_redemptions_count`,
  `concurrent_campaigns_count` all strongly higher for treated) — households already known
  to engage with campaigns and coupons were targeted again.
- **`has_demographics` is the second-largest gap** — treated households are far more likely
  to have a demographics record on file. Demographics are voluntary (Stage 1 found only
  ~32% coverage), so this strongly suggests the retailer targeted known/enrolled loyalty
  customers rather than anonymous card users.
- **Shopping intensity** (`monetary_spend`, `frequency_baskets` much higher, `recency_days`
  much lower for treated) — targeting favored heavy, frequent, recently-active shoppers.
- Several demographic categories (homeowners, married, specific age/income bands, no-kids
  households) also skew treated, consistent with demographic-based targeting rather than
  a random campaign rollout.

This is exactly why the uplift stage needs propensity score adjustment rather than a naive
treated-vs-control comparison: these imbalances mean a simple average outcome difference
would partly reflect "who tends to spend more regardless of the coupon" rather than the
coupon's actual effect. It also means the propensity model should score well — targeting is
predictable from pre-period behavior — and overlap between treated and control propensity
scores will need to be checked carefully before trusting the uplift estimates.
